# 07 — Retrieval-Augmented Generation (RAG)

Give the chatbot a document, and it answers **from that document** — quoted, cited, or refused when the answer isn't there. This notebook walks the whole Phase 7 path: chunk → embed → store → retrieve → grounded answer.

Production code: `app/rag/` · API: `/api/documents/*` · Docs: `docs/07_rag.md`

In [ ]:
import sys, tempfile, time
from pathlib import Path

cwd = Path.cwd()
backend_dir = cwd.parent / "backend" if cwd.name == "notebooks" else cwd / "backend"
sys.path.insert(0, str(backend_dir))

import numpy as np
from app.rag.chunker import chunk_text
from app.rag.store import RagStore, ScoredChunk, EMBEDDING_DIM
from app.rag.answerer import compose_grounded_answer
from app.services.embedding_service import get_embedding_service

t0 = time.time()
emb = get_embedding_service()
print(f"model: {emb.model_name}, dim={emb.dim} ({time.time()-t0:.1f}s)")

## 1. Chunking — sliding word window with overlap

Embedding models cap out around 256 words, and retrieval wants the *paragraph*, not the whole document. NOVA splits on an 80-word window stepping 65 words (15 of overlap) and keeps **exact character offsets** so citations can point back into the original file.

In [ ]:
document = (
    "Vacation policy: employees accrue 20 paid vacation days per year. "
    "Unused days roll over up to 5 days into the next calendar year. "
    "Requests longer than one week need manager approval submitted at least "
    "14 days in advance. Paid sick leave accrues separately at 10 days per year "
    "and does not roll over. Parental leave is 16 weeks at full pay. "
    "Remote work requires a signed equipment agreement. "
    "Travel reimbursements must be filed within 30 days of returning. "
    "Annual performance reviews happen in December and feed the bonus pool. "
    "New hires start with a 90-day probation period during which either party "
    "may end employment with one week of notice. "
    "The office is closed between Christmas and New Year's Eve."
) 

chunks = chunk_text(document, chunk_words=80, overlap_words=15)
print(f"{len(document)} chars -> {len(chunks)} chunks")
for c in chunks:
    assert document[c.start:c.end] == c.text  # offsets are exact
    print(f"  [{c.start:>4}:{c.end:<4}] {len(c.text.split()):>3} words | {c.text[:60]}...")

## 2. Why overlap matters

An answer can straddle a chunk boundary. Cut a sentence in half and *neither* half embeds like the original — retrieval quality collapses. Below: the same sentence embedded whole vs. split, with and without overlap.

In [ ]:
query = "how many vacation days do employees get"
answer_sentence = " employees accrue 20 paid vacation days per year "
before = "churn churn churn. " * 12          # filler pushes the answer across a boundary
after = " more filler words continue the document here"
text = before + answer_sentence + after

def halves(chunked):
    return [c.text for c in chunked]

no_overlap = chunk_text(text, chunk_words=20, overlap_words=0)
with_overlap = chunk_text(text, chunk_words=20, overlap_words=8)

q = emb.embed(query)
def best_sim(pieces):
    vecs = emb.encode(pieces)
    return float(np.max(vecs @ q))

print(f"no overlap  : best cosine {best_sim(halves(no_overlap)):.3f}")
print(f"with overlap: best cosine {best_sim(halves(with_overlap)):.3f}")
print("(overlap keeps the full answer sentence inside at least one chunk)")

## 3. Vector store — embed, persist, top-k search

Cosine similarity on unit vectors = dot product (`score = q @ C.T`). NOVA's `RagStore` is dual-mode: **pgvector** (`ORDER BY embedding_vec <=> $q` inside Postgres) when the extension is available, else the **numpy fallback** loaded from `bytea`. Same schema, same API — this cell runs on SQLite.

In [ ]:
tmp_db = Path(tempfile.mkdtemp()) / "rag_demo.db"
store = RagStore(f"sqlite:///{tmp_db}")
print("mode:", "pgvector" if store.vector_enabled else "numpy fallback")

pieces = chunk_text(document, chunk_words=40, overlap_words=8)
embeddings = emb.encode([p.text for p in pieces])
info = store.add_document(
    filename="handbook.txt",
    content_type="text/plain",
    size_bytes=len(document),
    stored_path=None,
    chunks=[p.text for p in pieces],
    embeddings=embeddings,
)
print(f"stored {info.filename}: {info.chunk_count} chunks, id={info.id[:8]}...")

hits = store.search(emb.embed("can I roll over unused vacation days"), top_k=3)
for h in hits:
    print(f"  {h.score:.3f} part {h.chunk_index + 1}: {h.content[:70]}...")

## 4. Grounding & refusal — the anti-hallucination contract

Three outcomes, all explicit:

1. **no documents** → ask for an upload
2. **weak scores** (< `NOVA_RAG_MIN_SCORE`, default 0.35) → *"I couldn't find that... so I won't guess"*
3. **clears the gate** → verbatim quote + `sources[]` citations

In [ ]:
question = "how many vacation days do employees get"
hits = store.search(emb.embed(question), top_k=3)

# 3) grounded: retrieval clears the gate
answer = compose_grounded_answer(question, hits, min_score=0.35, has_documents=True)
print("GROUNDED:", answer.grounded)
print(answer.reply)
print("sources:", [(s.filename, s.chunk_index, s.score) for s in answer.sources])
print()

# 2) refusal: pretend retrieval found nothing relevant
weak = [ScoredChunk(document_id=info.id, filename=info.filename, chunk_index=0,
                    content="unrelated boilerplate", score=0.12)]
refusal = compose_grounded_answer(question, weak, min_score=0.35, has_documents=True)
print("REFUSAL  :", refusal.grounded, "->", refusal.reply)
print()

# 1) no knowledge base at all
empty = compose_grounded_answer(question, [], min_score=0.35, has_documents=False)
print("NO DOCS  :", empty.grounded, "->", empty.reply)

## 5. From notebook to API

The same pipeline ships behind three endpoints:

```powershell
# ingest
curl -F "file=@nova\data\samples\employee_handbook.txt" http://localhost:8000/api/documents/upload
# ask — routed by the document_question intent
curl -X POST http://localhost:8000/api/chat -H "Content-Type: application/json" `
     -d '{"message":"what does the document say about vacation days"}'
```

**Exercises**
1. Vary `chunk_words`/`overlap_words` and watch retrieval scores move — find the
   setting where the answer sentence lands cleanly inside one chunk.
2. Swap the cosine gate: what `min_score` refuses the *unanswerable* question
   while still accepting the answerable one? (Overfitting the gate = the
   precision/recall trade-off in disguise.)
3. Read `app/rag/store.py`'s `_search_pgvector` — where would an `hnsw` index
   change the query plan?
4. Upload a real PDF and check the extracted text quality — what breaks first?